In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# === Paths ===
GSS_FILE    = Path(r"D:\Shikha\bias_project\validation_data\gss\gss_gender_attitudes.xlsx")
OUTPUT_DIR  = Path(r"D:\Shikha\bias_project\validation_data\gss\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Missing value codes used by GSS ===
# These must be treated as NaN before any analysis
MISSING_CODES = [-100, -99, -98, -97, -96, -90, -89, -70]

# === Load data ===
print("Loading GSS data...")
df = pd.read_excel(GSS_FILE, sheet_name='Data')

print(f"Rows loaded      : {len(df):,}")
print(f"Columns          : {list(df.columns)}")
print(f"\nSample (first 3 rows):")
print(df.head(3))

Loading GSS data...
Rows loaded      : 75,699
Columns          : ['year', 'id_', 'marital', 'age', 'educ', 'sex', 'race', 'reg16', 'region', 'fework', 'fepres', 'fepol', 'fechld', 'fehelp', 'fefam', 'fehire', 'ballot', 'REG16_7222', 'REGION_7222']

Sample (first 3 rows):
   year  id_  marital  age  educ  sex  race  reg16  region  fework  fepres  \
0  1972    1        5   23    16    2     1      1       2       1       1   
1  1972    2        1   70    10    1     1      2       2       2       2   
2  1972    3        1   48    12    2     1      2       2       2       1   

   fepol  fechld  fehelp  fefam  fehire  ballot  REG16_7222  REGION_7222  
0   -100    -100    -100   -100    -100    -100           2            3  
1   -100    -100    -100   -100    -100    -100           3            3  
2   -100    -100    -100   -100    -100    -100           3            3  


In [4]:
# === Replace missing codes with NaN ===
ATTITUDE_VARS = ['fefam', 'fepres', 'fework', 'fechld', 'fehelp', 'fepol', 'fehire']

df_clean = df.copy()
for col in ATTITUDE_VARS:
    df_clean[col] = df_clean[col].replace(MISSING_CODES, np.nan)

print("Missing value replacement done.")
print(f"Rows with at least one valid attitude response: "
      f"{df_clean[ATTITUDE_VARS].notna().any(axis=1).sum():,}")

# === Region-year aggregates ===
# REGION_7222 is the 9-category region variable (preferred)
# Group by year + region, compute mean of each attitude variable

region_year = (
    df_clean
    .groupby(['year', 'REGION_7222'])[ATTITUDE_VARS]
    .mean()
    .round(3)
    .reset_index()
)

region_year.columns = ['year', 'region', 'fefam', 'fepres', 'fework',
                        'fechld', 'fehelp', 'fepol', 'fehire']

print(f"\nRegion-year panel shape: {region_year.shape}")
print(f"Years covered: {region_year['year'].min()} – {region_year['year'].max()}")
print(f"Regions: {sorted(region_year['region'].unique())}")
print(f"\nSample:")
print(region_year.head(10))

# === Save ===
OUT_FILE = OUTPUT_DIR / "gss_region_year_means.csv"
region_year.to_csv(OUT_FILE, index=False)
print(f"\nSaved to: {OUT_FILE}")

Missing value replacement done.
Rows with at least one valid attitude response: 47,516

Region-year panel shape: (307, 9)
Years covered: 1972 – 2024
Regions: [np.int64(-70), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

Sample:
   year  region  fefam  fepres  fework  fechld  fehelp  fepol  fehire
0  1972       1    NaN   1.207   1.289     NaN     NaN    NaN     NaN
1  1972       2    NaN   1.210   1.305     NaN     NaN    NaN     NaN
2  1972       3    NaN   1.246   1.348     NaN     NaN    NaN     NaN
3  1972       4    NaN   1.270   1.449     NaN     NaN    NaN     NaN
4  1972       5    NaN   1.320   1.284     NaN     NaN    NaN     NaN
5  1972       6    NaN   1.301   1.392     NaN     NaN    NaN     NaN
6  1972       7    NaN   1.339   1.332     NaN     NaN    NaN     NaN
7  1972       8    NaN   1.339   1.482     NaN     NaN    NaN     NaN
8  1972       9    NaN   1.222   1.381     NaN     NaN    NaN     NaN


In [5]:
# === Filter out region -70 (not applicable) ===
region_year = region_year[region_year['region'] > 0].copy()

# === Drop rows where ALL attitude vars are NaN (no data for that region-year) ===
region_year = region_year.dropna(subset=ATTITUDE_VARS, how='all').reset_index(drop=True)

# === Add region labels ===
# GSS 9-category REGION_7222 coding
REGION_LABELS = {
    1: 'New England',
    2: 'Middle Atlantic',
    3: 'East North Central',
    4: 'West North Central',
    5: 'South Atlantic',
    6: 'East South Central',
    7: 'West South Central',
    8: 'Mountain',
    9: 'Pacific'
}
region_year['region_name'] = region_year['region'].map(REGION_LABELS)

# === Compute a composite gender attitude score ===
# Simple mean across all available attitude vars per row
# Higher score = more traditional/restrictive attitudes
region_year['composite_score'] = region_year[ATTITUDE_VARS].mean(axis=1).round(3)

print(f"Clean panel shape : {region_year.shape}")
print(f"Years covered     : {region_year['year'].min()} – {region_year['year'].max()}")
print(f"Regions           : {region_year['region_name'].unique()}")
print(f"\nSample:")
print(region_year[['year','region_name','fefam','fepres','fework','composite_score']].head(12))

# === Save cleaned version ===
OUT_CLEAN = OUTPUT_DIR / "gss_region_year_clean.csv"
region_year.to_csv(OUT_CLEAN, index=False)
print(f"\nSaved to: {OUT_CLEAN}")

Clean panel shape : (261, 11)
Years covered     : 1972 – 2022
Regions           : <ArrowStringArray>
[       'New England',    'Middle Atlantic', 'East North Central',
 'West North Central',     'South Atlantic', 'East South Central',
 'West South Central',           'Mountain',            'Pacific']
Length: 9, dtype: str

Sample:
    year         region_name  fefam  fepres  fework  composite_score
0   1972         New England    NaN   1.207   1.289            1.248
1   1972     Middle Atlantic    NaN   1.210   1.305            1.257
2   1972  East North Central    NaN   1.246   1.348            1.297
3   1972  West North Central    NaN   1.270   1.449            1.360
4   1972      South Atlantic    NaN   1.320   1.284            1.302
5   1972  East South Central    NaN   1.301   1.392            1.346
6   1972  West South Central    NaN   1.339   1.332            1.336
7   1972            Mountain    NaN   1.339   1.482            1.410
8   1972             Pacific    NaN   1.222   

In [6]:
# === Summary stats useful for Friday demo ===

# 1. How attitudes changed over time nationally
national_trend = (
    df_clean.groupby('year')[ATTITUDE_VARS]
    .mean()
    .round(3)
    .reset_index()
)
national_trend['composite'] = national_trend[ATTITUDE_VARS].mean(axis=1).round(3)

# 2. Most vs least traditional regions (averaged across all years)
region_avg = (
    region_year.groupby('region_name')['composite_score']
    .mean()
    .round(3)
    .sort_values(ascending=False)
    .reset_index()
)
region_avg.columns = ['region', 'avg_composite_score']

# 3. Coverage summary — how many region-year cells have each variable
coverage = {col: region_year[col].notna().sum() for col in ATTITUDE_VARS}

print("=== National trend (composite score, selected years) ===")
print(national_trend[['year','composite']].set_index('year').T.to_string())

print("\n=== Regions ranked by avg gender attitude (higher = more traditional) ===")
print(region_avg.to_string(index=False))

print("\n=== Variable coverage (cells with data out of 261) ===")
for var, count in coverage.items():
    print(f"  {var:<8} : {count} cells ({count/261*100:.0f}%)")

# Save national trend
national_trend.to_csv(OUTPUT_DIR / "gss_national_trend.csv", index=False)
print(f"\nAll outputs saved to: {OUTPUT_DIR}")

=== National trend (composite score, selected years) ===
year        1972  1973   1974   1975  1976   1977   1978  1980   1982   1983  1984   1985  1986  1987   1988   1989   1990   1991   1993   1994   1996   1998   2000   2002   2004  2006   2008   2010   2012   2014  2016   2018   2021   2022   2024
composite  1.305   NaN  1.345  1.332   NaN  1.859  1.337   NaN  1.339  1.338   NaN  1.914  1.92   NaN  1.951  1.964  1.964  1.964  1.993  1.987  2.078  1.997  2.289  2.269  2.285  2.27  2.011  2.001  2.284  2.251  2.24  2.266  2.466  2.288  2.287

=== Regions ranked by avg gender attitude (higher = more traditional) ===
            region  avg_composite_score
West North Central                1.979
       New England                1.975
          Mountain                1.971
           Pacific                1.968
East North Central                1.959
   Middle Atlantic                1.939
    South Atlantic                1.938
West South Central                1.918
East South Cen

In [7]:
print(OUTPUT_DIR)
print(OUTPUT_DIR.exists())
import os
print(os.listdir(OUTPUT_DIR))

D:\Shikha\bias_project\validation_data\gss\processed
True
['gss_national_trend.csv', 'gss_region_year_clean.csv', 'gss_region_year_means.csv']
